# SmishTank preparation for Dataset 1

This notebook prepares `Smishtank Dataset I.csv` for the v2 Core human-side gathering stage.

Source taxonomy: the CSV column **`Message Categories`** (not renamed in this pipeline) is copied into `message_categories` after whitespace strip only — **no coercion or relabeling** of the source category string.

Per `v2/docs/dataset_design_final.md` §5.7–5.8, **`sms_fraud_subtype`** is the separate Core axis used for inclusion policy (`core_keep_for_dataset1`). Here it is derived **only** by a fixed lookup from `message_categories` (no text-based overrides). Manual remapping of `Advertisement` rows by content is a **separate review step**, not done in this notebook.

Pipeline:
- read and clean `MainText`,
- apply reproducible English-like filtering,
- URL masking and deduplication,
- set `sms_fraud_subtype` from source category via fixed table; set `core_keep_for_dataset1` per §5.8,
- **export to gathered only rows with `core_keep_for_dataset1 == "yes"`** (Core §5.8 definite include: `account_alert`, `delivery_fee_or_service_issue`, `prize_or_contest_scam`). Rows with `review` or `no` are excluded from `gathered` — they are not written to `v2/data/interim/gathered/` (inspect counts in the notebook).


In [1]:
from __future__ import annotations

import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd
from langdetect import detect, LangDetectException, DetectorFactory

DetectorFactory.seed = 0  # reproducibility

BASE = Path('/Users/askar/projects/antifraud-deepfake-detection/v2')
sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin  # channel-level thresholds from 00_length_bin_analysis
RAW_PATH = BASE / 'data/raw/Smishtank Dataset I.csv'
OUT_DIR = BASE / 'data/interim/gathered'
OUT_PATH = OUT_DIR / 'smishtank_prepared.jsonl'

# Core ontology for `sms_fraud_subtype` (§5.7 dataset_design_final.md).
ALLOWED_SUBTYPES = {
    'account_alert',
    'delivery_fee_or_service_issue',
    'prize_or_contest_scam',
    'financial_or_crypto_lure',
    'loan_or_credit_lure',
    'generic_deceptive_sms',
    'wrong_number_or_romance_scam',
    'other_or_unclear',
}

# Fixed 1:1 from source `Message Categories` → Core `sms_fraud_subtype`.
# Does not modify `message_categories`; unknown future labels → other_or_unclear.
MESSAGE_CATEGORY_TO_SUBTYPE = {
    'Account Alert': 'account_alert',
    'Delivery': 'delivery_fee_or_service_issue',
    'Prize/Contest': 'prize_or_contest_scam',
    'Finance/Crypto': 'financial_or_crypto_lure',
    'Loans/Credit': 'loan_or_credit_lure',
    'Wrong Number/Romance Scam': 'wrong_number_or_romance_scam',
    'Advertisement': 'generic_deceptive_sms',
    'Job Advertisement': 'generic_deceptive_sms',
    'Lawsuits/Settlements': 'generic_deceptive_sms',
    'Other': 'other_or_unclear',
}

URL_RE = re.compile(r'(?i)(?:https?://\S+|www\.\S+|\b[a-z0-9][a-z0-9-]*\.[a-z]{2,}(?:/\S*)?)')
EMAIL_RE = re.compile(r'(?i)\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b')
PHONE_RE = re.compile(r'(?:(?:\+?\d[\d\-\s()]{6,}\d)|(?:\b\d{5,}\b))')
TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)


def normalize_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\uFFFD', ' ')
    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def mask_url(text: str) -> str:
    return URL_RE.sub('[URL]', text)


def is_english(text: str) -> bool:
    """Detect English via langdetect (seed=0 for reproducibility)."""
    if not text or len(text.strip()) < 15:
        return False
    try:
        return detect(text[:2000]) == 'en'
    except LangDetectException:
        return False


def to_yes_no(v: bool) -> str:
    return 'yes' if v else 'no'


def token_length(text: str) -> int:
    return len(TOKEN_RE.findall(text))


def subtype_from_source_category(message_category: str) -> str:
    mc = (message_category or '').strip()
    return MESSAGE_CATEGORY_TO_SUBTYPE.get(mc, 'other_or_unclear')


def core_policy_flag(subtype: str) -> str:
    if subtype in {'account_alert', 'delivery_fee_or_service_issue', 'prize_or_contest_scam'}:
        return 'yes'
    if subtype in {'financial_or_crypto_lure', 'loan_or_credit_lure'}:
        return 'review'
    return 'no'


In [2]:
# Load, preprocess, filter language, and deduplicate
df = pd.read_csv(RAW_PATH, encoding='latin-1', keep_default_na=False)

df['message_categories'] = df['Message Categories'].astype(str).str.strip()
df['text_raw'] = df['MainText'].astype(str)
df['text_norm'] = df['text_raw'].map(normalize_text)
df = df[df['text_norm'] != ''].copy()

df['text'] = df['text_norm'].map(mask_url).map(normalize_text)
df = df[df['text'] != ''].copy()

df['lang_filter_method'] = 'langdetect_v1'
df['is_english'] = df['text'].map(is_english)
df_en = df[df['is_english']].copy()

df_en['has_url'] = df_en['text_raw'].str.contains(URL_RE, regex=True)
df_en['has_email'] = df_en['text_raw'].str.contains(EMAIL_RE, regex=True)
df_en['has_phone_or_reply_cta'] = df_en['text_raw'].str.contains(PHONE_RE, regex=True) | df_en['text_raw'].str.contains(r'(?i)\b(?:reply|call|text|sms|whatsapp)\b', regex=True)

grp = df_en.groupby('text', sort=False)
prepared = grp.first().reset_index(drop=False)
prepared['n_duplicate_rows'] = grp.size().values

# Preserve source categories verbatim (strip only, applied when building df).
prepared['sms_fraud_subtype'] = prepared['message_categories'].map(subtype_from_source_category)
prepared['core_keep_for_dataset1'] = prepared['sms_fraud_subtype'].map(core_policy_flag)

assert set(prepared['sms_fraud_subtype']).issubset(ALLOWED_SUBTYPES)

prepared['char_length'] = prepared['text'].str.len()
prepared['token_length'] = prepared['text'].map(token_length)
# length_bin computed from channel-level thresholds (v2/config.py, §5.1).
# Audit finding Apr-2026: hardcoded 'short' was wrong for 37% of rows (40–197 tokens → medium).
prepared['length_bin'] = prepared['token_length'].map(lambda t: compute_length_bin(t, 'sms'))
prepared['time_band'] = 'legacy'

prepared['label'] = 0
prepared['label_str'] = 'human'
prepared['fraudness'] = 'fraud'
prepared['channel'] = 'sms'
prepared['scenario_family'] = 'fraud_sms_deceptive'
prepared['source_family'] = 'smishtank'
prepared['dataset_source'] = 'Smishtank Dataset I.csv'
prepared['origin_model'] = 'human'
prepared['split'] = 'tbd'

prepared['has_url'] = prepared['has_url'].map(to_yes_no)
prepared['has_email'] = prepared['has_email'].map(to_yes_no)
prepared['has_phone_or_reply_cta'] = prepared['has_phone_or_reply_cta'].map(to_yes_no)

out_cols = [
    'text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family',
    'source_family', 'dataset_source', 'message_categories', 'sms_fraud_subtype',
    'core_keep_for_dataset1', 'has_url', 'has_phone_or_reply_cta', 'has_email',
    'char_length', 'token_length', 'length_bin', 'time_band',
    'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows'
]
prepared_out = prepared[out_cols].copy()

# Gathered = Dataset 1 candidates only: Core §5.8 "включать точно" (core_keep_for_dataset1 == yes).
gathered_out = prepared_out[prepared_out['core_keep_for_dataset1'] == 'yes'].copy()

OUT_DIR.mkdir(parents=True, exist_ok=True)
gathered_out.to_json(OUT_PATH, orient='records', lines=True, force_ascii=False)

print('raw rows:', len(pd.read_csv(RAW_PATH, encoding='latin-1', keep_default_na=False)))
print('after cleaning non-empty text:', len(df))
print('after english filter:', len(df_en))
print('after dedup (all candidates):', len(prepared_out))
print('after Core policy filter (written to gathered):', len(gathered_out))
print('excluded review:', (prepared_out['core_keep_for_dataset1'] == 'review').sum())
print('excluded no:', (prepared_out['core_keep_for_dataset1'] == 'no').sum())
print('output:', OUT_PATH)


raw rows: 1062
after cleaning non-empty text: 1062
after english filter: 1007
after dedup (all candidates): 936
after Core policy filter (written to gathered): 483
excluded review: 61
excluded no: 392
output: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/smishtank_prepared.jsonl


In [3]:
# Validation and distributions
check_df = pd.read_json(OUT_PATH, lines=True)

assert check_df['text'].astype(str).str.strip().ne('').all()
assert check_df['message_categories'].astype(str).str.strip().ne('').all()
assert check_df['sms_fraud_subtype'].astype(str).str.strip().ne('').all()
assert set(check_df['sms_fraud_subtype']).issubset(ALLOWED_SUBTYPES)
assert (check_df['core_keep_for_dataset1'] == 'yes').all(), 'gathered file must contain only Core-definite-includes'
assert set(check_df['length_bin']).issubset({'short', 'medium', 'long'}), 'unexpected length_bin values'

unknown_cats = sorted(check_df.loc[~check_df['message_categories'].isin(MESSAGE_CATEGORY_TO_SUBTYPE), 'message_categories'].unique())
print('Source categories outside fixed lookup table (should be empty for current CSV):', unknown_cats)

print('Rows in gathered JSONL (Core yes only):', len(check_df))
print('\nmessage_categories distribution:')
print(check_df['message_categories'].value_counts())
print('\nsms_fraud_subtype distribution:')
print(check_df['sms_fraud_subtype'].value_counts())
print('\ncore_keep_for_dataset1 distribution:')
print(check_df['core_keep_for_dataset1'].value_counts())
print('\nlength_bin distribution:')
print(check_df['length_bin'].value_counts())

print('\nFiles in gathered:')
for p in sorted(OUT_DIR.glob('*')):
    print('-', p.name)

non_jsonl = [p.name for p in OUT_DIR.glob('*') if p.is_file() and p.suffix.lower() != '.jsonl']
print('\nNon-JSONL files in gathered:', non_jsonl if non_jsonl else 'none')


Source categories outside fixed lookup table (should be empty for current CSV): []
Rows in gathered JSONL (Core yes only): 483

message_categories distribution:
message_categories
Account Alert    281
Delivery         148
Prize/Contest     54
Name: count, dtype: int64

sms_fraud_subtype distribution:
sms_fraud_subtype
account_alert                    281
delivery_fee_or_service_issue    148
prize_or_contest_scam             54
Name: count, dtype: int64

core_keep_for_dataset1 distribution:
core_keep_for_dataset1
yes    483
Name: count, dtype: int64

length_bin distribution:
length_bin
medium    357
long      100
short      26
Name: count, dtype: int64

Files in gathered:
- enron_ham_prepared.jsonl
- financial_qa_prepared.jsonl
- mendeley_smishing_prepared.jsonl
- nazario_prepared.jsonl
- nigerian_fraud_prepared.jsonl
- smishtank_prepared.jsonl
- sms_ham_prepared.jsonl
- spamassassin_ham_prepared.jsonl

Non-JSONL files in gathered: none
